# 🗺️ Cartographie Sémantique — Espace Latent ECG (UMAP + Plotly)

**Objectif** : Projeter nos vecteurs OpenAI `text-embedding-3-small` (1536 dimensions) en 2D via **UMAP**,
puis tracer une carte interactive montrant :
- 🔵 **L'ontologie ECG** (522 documents = concepts canoniques + synonymes)
- ⭐ **Les termes étudiants** (extraits des réponses des 5 cardiologues)

| Cellule | Description |
|---------|-------------|
| 1 | **Imports & Setup** — numpy, umap-learn, plotly, OpenAI client |
| 2 | **Fond de Carte** — Chargement vecteurs + métadonnées ontologie |
| 3 | **Explorateurs** — Extraction & vectorisation des termes étudiants |
| 4 | **UMAP** — Réduction 1536D → 2D (fit sur ontologie, transform étudiants) |
| 5 | **Cartographie** — Scatter interactif Plotly (hover = terme médical) |

In [1]:
# ============================================================
# CELLULE 1 — Imports & Setup
# ============================================================
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import os

# UMAP pour la réduction de dimensionnalité (fallback t-SNE si besoin)
try:
    import umap
    REDUCER_NAME = 'UMAP'
    print(f"[OK] umap-learn {umap.__version__} chargé")
except ImportError:
    from sklearn.manifold import TSNE
    REDUCER_NAME = 't-SNE'
    print("[WARN] umap-learn non disponible, fallback sur t-SNE")

# Plotly pour la visualisation interactive
import plotly.express as px
import plotly.graph_objects as go

# Client OpenAI pour vectoriser les termes étudiants
from openai import OpenAI

warnings.filterwarnings('ignore', category=FutureWarning)

# ─── Chemins racines ──────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\Administrateur\bmad\ECG lecture")
EVAL_ROOT    = Path(r"C:\Users\Administrateur\bmad\ECG evaluation")
RAG_ROOT     = Path(r"C:\Users\Administrateur\bmad\RAG ontologique")
INDEX_DIR    = RAG_ROOT / "rag_index"

# Charger la clé API OpenAI
load_dotenv(PROJECT_ROOT / ".env")
client = OpenAI()

EMBEDDING_MODEL = "text-embedding-3-small"  # 1536 dims

print(f"[OK] OPENAI_API_KEY : {'✅' if os.getenv('OPENAI_API_KEY') else '❌'}")
print(f"[OK] Réducteur      : {REDUCER_NAME}")
print(f"[OK] Index dir      : {INDEX_DIR}")
print(f"\n🚀 Setup terminé.")

c:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[OK] umap-learn 0.5.11 chargé
[OK] OPENAI_API_KEY : ✅
[OK] Réducteur      : UMAP
[OK] Index dir      : C:\Users\Administrateur\bmad\RAG ontologique\rag_index

🚀 Setup terminé.
[OK] OPENAI_API_KEY : ✅
[OK] Réducteur      : UMAP
[OK] Index dir      : C:\Users\Administrateur\bmad\RAG ontologique\rag_index

🚀 Setup terminé.


In [2]:
# ============================================================
# CELLULE 2 — Fond de Carte : Chargement de l'Ontologie
# ============================================================
# Matrice des embeddings OpenAI (N × 1536)
embeddings_onto = np.load(INDEX_DIR / "vecteurs_ontologie.npy")
print(f"[ONTO] Matrice embeddings : {embeddings_onto.shape} ({embeddings_onto.dtype})")

# Métadonnées : label, ID OWL, catégorie, type de surface form
with open(INDEX_DIR / "metadata_ontologie.json", 'r', encoding='utf-8') as f:
    meta_onto = json.load(f)

documents = meta_onto['documents']
assert len(documents) == embeddings_onto.shape[0], "Incohérence taille embeddings / métadonnées"

# Construire un DataFrame pour l'ontologie
df_onto = pd.DataFrame(documents)
df_onto['type'] = '🔵 Ontologie'
df_onto['label'] = df_onto['surface_form']
df_onto['hover'] = df_onto.apply(
    lambda r: f"{r['surface_form']}\n[{r['ontology_id']}]\n{r['categorie']} ({r['source_type']})",
    axis=1
)

# Stats
print(f"[ONTO] {len(df_onto)} documents chargés")
print(f"\n   Répartition par catégorie :")
for cat, count in df_onto['categorie'].value_counts().items():
    print(f"      {cat:<35s} : {count}")
print(f"\n   Répartition par type :")
for st, count in df_onto['source_type'].value_counts().items():
    print(f"      {st:<15s} : {count}")

print(f"\n✅ Fond de carte prêt : {len(df_onto)} points à projeter.")

[ONTO] Matrice embeddings : (537, 1536) (float32)
[ONTO] 537 documents chargés

   Répartition par catégorie :
      DESCRIPTEUR_ECG                     : 223
      DIAGNOSTIC_MAJEUR                   : 128
      SIGNE_ECG_PATHOLOGIQUE              : 126
      DIAGNOSTIC_URGENT                   : 60

   Répartition par type :
      canonical       : 292
      synonym         : 245

✅ Fond de carte prêt : 537 points à projeter.


In [3]:
# ============================================================
# CELLULE 3 — Explorateurs : Termes Étudiants Vectorisés
# ============================================================

# ─── A) Charger les réponses brutes des collègues ────────────
df_responses = pd.read_csv(EVAL_ROOT / "ECG_Collector_Data.csv")
PARTICIPANTS = df_responses['code'].tolist()
cas_cols = [c for c in df_responses.columns if c.startswith('cas_')]

print(f"[CSV] {len(PARTICIPANTS)} participants, {len(cas_cols)} cas")

# ─── B) Extraire des termes cliniques uniques ────────────────
# On prend les phrases brutes et on les découpe en segments courts
# (séparés par virgule, retour ligne, ou point)
import re

all_terms = set()
for _, row in df_responses.iterrows():
    for col in cas_cols:
        text = str(row[col]).strip()
        if text and text != 'nan':
            # Découper en segments
            segments = re.split(r'[,\n.;]+', text)
            for seg in segments:
                seg = seg.strip()
                if len(seg) >= 3 and len(seg) <= 100:  # Filtrer les trop courts/longs
                    all_terms.add(seg)

# Dédupliquer et trier
student_terms = sorted(all_terms)
print(f"[ETU] {len(student_terms)} termes cliniques uniques extraits")
print(f"   Exemples : {student_terms[:8]}")

# ─── C) Vectoriser via OpenAI text-embedding-3-small ─────────
print(f"\n⏳ Vectorisation de {len(student_terms)} termes via {EMBEDDING_MODEL}...")

# Batch par lots de 100 (limite API)
BATCH_SIZE = 100
all_embeddings = []

for i in range(0, len(student_terms), BATCH_SIZE):
    batch = student_terms[i:i+BATCH_SIZE]
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=batch
    )
    batch_embs = [item.embedding for item in response.data]
    all_embeddings.extend(batch_embs)
    print(f"   Batch {i//BATCH_SIZE + 1}/{(len(student_terms)-1)//BATCH_SIZE + 1} — {len(batch)} termes")

embeddings_students = np.array(all_embeddings, dtype=np.float32)
print(f"\n[ETU] Matrice embeddings étudiants : {embeddings_students.shape}")
print(f"✅ {len(student_terms)} termes vectorisés.")

[CSV] 5 participants, 15 cas
[ETU] 195 termes cliniques uniques extraits
   Exemples : ['150 bpm', '60 bpm', '62 bpm', '62bpm', '70 bpm', '75 bpm', '78 bpm', '80 bpm']

⏳ Vectorisation de 195 termes via text-embedding-3-small...
   Batch 1/2 — 100 termes
   Batch 1/2 — 100 termes
   Batch 2/2 — 95 termes

[ETU] Matrice embeddings étudiants : (195, 1536)
✅ 195 termes vectorisés.
   Batch 2/2 — 95 termes

[ETU] Matrice embeddings étudiants : (195, 1536)
✅ 195 termes vectorisés.


In [4]:
# ============================================================
# CELLULE 4 — Réduction UMAP : 1536D → 2D
# ============================================================
# Stratégie :
#   1. fit_transform() sur l'ontologie (= espace de référence)
#   2. transform() des étudiants sur cet espace (sans le déformer)

print(f"⏳ Réduction {REDUCER_NAME} : {embeddings_onto.shape[1]}D → 2D")
print(f"   Ontologie  : {embeddings_onto.shape[0]} points")
print(f"   Étudiants  : {embeddings_students.shape[0]} points")

if REDUCER_NAME == 'UMAP':
    # UMAP avec métrique cosinus (naturelle pour les embeddings)
    reducer = umap.UMAP(
        n_components=2,
        metric='cosine',
        n_neighbors=15,
        min_dist=0.1,
        random_state=42,
        verbose=True,
    )
    # 1. Entraîner sur l'ontologie (espace de référence)
    coords_onto = reducer.fit_transform(embeddings_onto)
    # 2. Projeter les étudiants sur le même espace
    coords_students = reducer.transform(embeddings_students)
else:
    # Fallback t-SNE : on concatène tout et on projette ensemble
    all_embs = np.vstack([embeddings_onto, embeddings_students])
    tsne = TSNE(n_components=2, metric='cosine', random_state=42, perplexity=30)
    coords_all = tsne.fit_transform(all_embs)
    coords_onto = coords_all[:len(embeddings_onto)]
    coords_students = coords_all[len(embeddings_onto):]

print(f"\n[OK] Ontologie  → {coords_onto.shape}")
print(f"[OK] Étudiants  → {coords_students.shape}")

# ─── Assembler le DataFrame final ────────────────────────────
df_plot_onto = pd.DataFrame({
    'x': coords_onto[:, 0],
    'y': coords_onto[:, 1],
    'label': df_onto['surface_form'].values,
    'type': '🔵 Ontologie',
    'categorie': df_onto['categorie'].values,
    'source': df_onto['source_type'].values,
    'concept_id': df_onto['ontology_id'].values,
})

df_plot_students = pd.DataFrame({
    'x': coords_students[:, 0],
    'y': coords_students[:, 1],
    'label': student_terms,
    'type': '⭐ Étudiant',
    'categorie': 'RÉPONSE_ÉTUDIANT',
    'source': 'student',
    'concept_id': '',
})

df_plot = pd.concat([df_plot_onto, df_plot_students], ignore_index=True)
print(f"\n✅ DataFrame combiné : {len(df_plot)} points ({len(df_plot_onto)} onto + {len(df_plot_students)} étudiants)")

⏳ Réduction UMAP : 1536D → 2D
   Ontologie  : 537 points
   Étudiants  : 195 points
UMAP(angular_rp_forest=True, metric='cosine', n_jobs=1, random_state=42, verbose=True)
Sat Feb 28 17:20:52 2026 Construct fuzzy simplicial set


c:\Python314\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Sat Feb 28 17:20:52 2026 Finding Nearest Neighbors
Sat Feb 28 17:20:54 2026 Finished Nearest Neighbor Search
Sat Feb 28 17:20:54 2026 Finished Nearest Neighbor Search
Sat Feb 28 17:20:56 2026 Construct embedding
Sat Feb 28 17:20:56 2026 Construct embedding


Epochs completed:  31%| ███        154/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs


Epochs completed: 100%| ██████████ 500/500 [00:00]



	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Sat Feb 28 17:20:57 2026 Finished embedding


Epochs completed: 100%| ██████████ 100/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs

[OK] Ontologie  → (537, 2)
[OK] Étudiants  → (195, 2)

✅ DataFrame combiné : 732 points (537 onto + 195 étudiants)


In [5]:
# ============================================================
# CELLULE 5 — Cartographie Interactive Plotly
# ============================================================

# ─── Palette de couleurs par catégorie ────────────────────────
COLOR_MAP = {
    'DIAGNOSTIC_MAJEUR':        '#2196F3',  # Bleu
    'DIAGNOSTIC_URGENT':        '#F44336',  # Rouge
    'SIGNE_ECG_PATHOLOGIQUE':   '#FF9800',  # Orange
    'DESCRIPTEUR_ECG':          '#4CAF50',  # Vert
    'RÉPONSE_ÉTUDIANT':         '#FFD700',  # Or (étoiles)
}

# Séparer ontologie et étudiants pour contrôler les marqueurs
df_onto_plot = df_plot[df_plot['type'] == '🔵 Ontologie']
df_etu_plot  = df_plot[df_plot['type'] == '⭐ Étudiant']

fig = go.Figure()

# ─── A) Points Ontologie (petits cercles colorés par catégorie) ─
for cat in ['DIAGNOSTIC_MAJEUR', 'DIAGNOSTIC_URGENT', 'SIGNE_ECG_PATHOLOGIQUE', 'DESCRIPTEUR_ECG']:
    mask = df_onto_plot['categorie'] == cat
    subset = df_onto_plot[mask]
    if len(subset) == 0:
        continue
    fig.add_trace(go.Scatter(
        x=subset['x'],
        y=subset['y'],
        mode='markers',
        marker=dict(
            size=5,
            color=COLOR_MAP[cat],
            opacity=0.6,
            line=dict(width=0.5, color='#333'),
        ),
        name=f"Onto: {cat.replace('_', ' ').title()}",
        text=subset['label'],
        customdata=np.stack([subset['concept_id'], subset['source']], axis=-1),
        hovertemplate=(
            "<b>%{text}</b><br>"
            "ID: %{customdata[0]}<br>"
            "Type: %{customdata[1]}<br>"
            f"Catégorie: {cat}<br>"
            "<extra></extra>"
        ),
    ))

# ─── B) Points Étudiants (grosses étoiles dorées) ─────────────
fig.add_trace(go.Scatter(
    x=df_etu_plot['x'],
    y=df_etu_plot['y'],
    mode='markers',
    marker=dict(
        size=9,
        color=COLOR_MAP['RÉPONSE_ÉTUDIANT'],
        symbol='star',
        opacity=0.85,
        line=dict(width=1, color='#B8860B'),
    ),
    name='⭐ Réponses Étudiants',
    text=df_etu_plot['label'],
    hovertemplate=(
        "<b>%{text}</b><br>"
        "Type: Réponse étudiant<br>"
        "<extra></extra>"
    ),
))

# ─── C) Mise en page (dark theme) ─────────────────────────────
fig.update_layout(
    title=dict(
        text="🗺️ Cartographie Sémantique de l'Ontologie ECG vs Réponses Étudiants",
        font=dict(size=18, color='#e0e0e0'),
        x=0.5,
    ),
    xaxis=dict(
        title=f"{REDUCER_NAME} Dimension 1",
        showgrid=True,
        gridcolor='#333',
        zeroline=False,
    ),
    yaxis=dict(
        title=f"{REDUCER_NAME} Dimension 2",
        showgrid=True,
        gridcolor='#333',
        zeroline=False,
    ),
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    font=dict(color='#e0e0e0'),
    legend=dict(
        bgcolor='#2d2d2d',
        bordercolor='#444',
        borderwidth=1,
        font=dict(size=11),
    ),
    width=1100,
    height=750,
    hovermode='closest',
)

fig.show()

print(f"\n📊 Carte tracée : {len(df_onto_plot)} points ontologie + {len(df_etu_plot)} termes étudiants")
print(f"   Réducteur : {REDUCER_NAME} (metric=cosine)")
print(f"   💡 Survolez les points pour voir le terme médical exact.")


📊 Carte tracée : 537 points ontologie + 195 termes étudiants
   Réducteur : UMAP (metric=cosine)
   💡 Survolez les points pour voir le terme médical exact.


In [6]:

# ============================================================
# CELLULE 6 — Lignes de Distance : Étudiant → Concept le + proche
# ============================================================
# On travaille dans l'espace ORIGINAL 1536D (pas le UMAP 2D)
# pour calculer la vraie similarité cosinus, puis on trace
# les lignes dans l'espace UMAP 2D pour la visualisation.

from sklearn.metrics.pairwise import cosine_similarity

# ─── A) Similarité cosinus étudiants × ontologie (195 × 522) ─
sim_matrix = cosine_similarity(embeddings_students, embeddings_onto)
print(f"[SIM] Matrice similarité : {sim_matrix.shape}")

# Pour chaque étudiant : index du concept ontologie le + proche
nn_indices = sim_matrix.argmax(axis=1)       # (195,)
nn_scores  = sim_matrix.max(axis=1)           # (195,) — score cosinus [0,1]
nn_distances = 1.0 - nn_scores                # distance = 1 - similarité

# ─── B) Construire le tableau de correspondances ──────────────
df_nn = pd.DataFrame({
    'terme_etudiant':     student_terms,
    'concept_onto':       [df_onto.iloc[i]['surface_form'] for i in nn_indices],
    'ontology_id':        [df_onto.iloc[i]['ontology_id'] for i in nn_indices],
    'categorie':          [df_onto.iloc[i]['categorie'] for i in nn_indices],
    'cosine_similarity':  nn_scores,
    'distance':           nn_distances,
    # Coordonnées UMAP de l'étudiant
    'etu_x': coords_students[:, 0],
    'etu_y': coords_students[:, 1],
    # Coordonnées UMAP du concept ontologie le + proche
    'onto_x': coords_onto[nn_indices, 0],
    'onto_y': coords_onto[nn_indices, 1],
})

# Stats
print(f"\n📊 Statistiques de distance (1 - cosine similarity) :")
print(f"   Moyenne   : {nn_distances.mean():.4f}")
print(f"   Médiane   : {np.median(nn_distances):.4f}")
print(f"   Min       : {nn_distances.min():.4f}  ({student_terms[nn_distances.argmin()]} → {df_nn.iloc[nn_distances.argmin()]['concept_onto']})")
print(f"   Max       : {nn_distances.max():.4f}  ({student_terms[nn_distances.argmax()]} → {df_nn.iloc[nn_distances.argmax()]['concept_onto']})")

# Seuils qualitatifs
n_excellent = (nn_scores >= 0.85).sum()
n_good      = ((nn_scores >= 0.70) & (nn_scores < 0.85)).sum()
n_medium    = ((nn_scores >= 0.50) & (nn_scores < 0.70)).sum()
n_poor      = (nn_scores < 0.50).sum()
print(f"\n   🟢 Excellent (sim ≥ 0.85) : {n_excellent} termes ({100*n_excellent/len(nn_scores):.0f}%)")
print(f"   🟡 Bon      (0.70–0.85)   : {n_good} termes ({100*n_good/len(nn_scores):.0f}%)")
print(f"   🟠 Moyen    (0.50–0.70)   : {n_medium} termes ({100*n_medium/len(nn_scores):.0f}%)")
print(f"   🔴 Faible   (sim < 0.50)  : {n_poor} termes ({100*n_poor/len(nn_scores):.0f}%)")

# ─── C) Carte avec lignes de distance ─────────────────────────
import plotly.graph_objects as go

fig2 = go.Figure()

# C.1) Fond : Points Ontologie (petits cercles gris discrets)
fig2.add_trace(go.Scatter(
    x=df_plot_onto['x'], y=df_plot_onto['y'],
    mode='markers',
    marker=dict(size=3, color='#666', opacity=0.3),
    name='Ontologie (fond)',
    text=df_plot_onto['label'],
    hovertemplate="<b>%{text}</b><extra>Ontologie</extra>",
))

# C.2) Lignes : étudiant → concept le + proche, colorées par distance
#   On utilise une échelle continue : vert (proche) → jaune → rouge (loin)
for _, row in df_nn.iterrows():
    sim = row['cosine_similarity']
    # Couleur : interpolation vert → jaune → rouge
    if sim >= 0.75:
        # Vert → Jaune  (sim 1.0→0.75)
        t = (1.0 - sim) / 0.25  # 0→1
        r = int(t * 255)
        g = 200
        b = 0
    else:
        # Jaune → Rouge (sim 0.75→0.0)
        t = (0.75 - sim) / 0.75  # 0→1
        r = 255
        g = int((1 - t) * 200)
        b = 0
    color = f'rgb({r},{g},{b})'

    fig2.add_trace(go.Scatter(
        x=[row['etu_x'], row['onto_x']],
        y=[row['etu_y'], row['onto_y']],
        mode='lines',
        line=dict(color=color, width=1.2),
        opacity=0.6,
        showlegend=False,
        hoverinfo='skip',
    ))

# C.3) Points cibles ontologie (les concepts matchés, en couleur)
matched_onto = df_nn.drop_duplicates(subset='ontology_id')
fig2.add_trace(go.Scatter(
    x=matched_onto['onto_x'], y=matched_onto['onto_y'],
    mode='markers',
    marker=dict(
        size=7,
        color=[COLOR_MAP.get(c, '#999') for c in matched_onto['categorie']],
        opacity=0.8,
        line=dict(width=1, color='white'),
        symbol='circle',
    ),
    name='Concepts matchés',
    text=matched_onto['concept_onto'],
    customdata=matched_onto['ontology_id'],
    hovertemplate="<b>%{text}</b><br>ID: %{customdata}<extra>Concept ontologie</extra>",
))

# C.4) Étoiles étudiants colorées par score
fig2.add_trace(go.Scatter(
    x=df_nn['etu_x'], y=df_nn['etu_y'],
    mode='markers',
    marker=dict(
        size=10,
        color=df_nn['cosine_similarity'],
        colorscale=[[0, '#F44336'], [0.5, '#FF9800'], [0.75, '#FFEB3B'], [1, '#4CAF50']],
        cmin=0.3, cmax=1.0,
        symbol='star',
        line=dict(width=1, color='#333'),
        colorbar=dict(
            title=dict(text='Similarité<br>cosinus', font=dict(size=11)),
            thickness=15,
            len=0.6,
            tickvals=[0.4, 0.6, 0.8, 1.0],
            ticktext=['0.4', '0.6', '0.8', '1.0'],
        ),
    ),
    name='⭐ Termes étudiants',
    text=df_nn['terme_etudiant'],
    customdata=np.stack([
        df_nn['concept_onto'],
        df_nn['cosine_similarity'].round(3).astype(str),
        df_nn['categorie'],
    ], axis=-1),
    hovertemplate=(
        "<b>⭐ %{text}</b><br>"
        "→ Concept : <b>%{customdata[0]}</b><br>"
        "Similarité : %{customdata[1]}<br>"
        "Catégorie : %{customdata[2]}<br>"
        "<extra></extra>"
    ),
))

# C.5) Layout dark theme
fig2.update_layout(
    title=dict(
        text="🎯 Distance Sémantique : Termes Étudiants → Concept Ontologie le + Proche",
        font=dict(size=16, color='#e0e0e0'),
        x=0.5,
    ),
    xaxis=dict(title=f"{REDUCER_NAME} Dim 1", showgrid=True, gridcolor='#333', zeroline=False),
    yaxis=dict(title=f"{REDUCER_NAME} Dim 2", showgrid=True, gridcolor='#333', zeroline=False),
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    font=dict(color='#e0e0e0'),
    legend=dict(bgcolor='#2d2d2d', bordercolor='#444', borderwidth=1, font=dict(size=11)),
    width=1100, height=800,
    hovermode='closest',
)

fig2.show()

print(f"\n🎯 Carte des distances tracée.")
print(f"   Chaque ligne relie un terme étudiant (⭐) à son concept ontologie le + proche (●).")
print(f"   🟢 Vert = proche (bon match)  →  🔴 Rouge = loin (terme non couvert par l'ontologie)")


[SIM] Matrice similarité : (195, 537)

📊 Statistiques de distance (1 - cosine similarity) :
   Moyenne   : 0.2677
   Médiane   : 0.2826
   Min       : -0.0000  (Bloc de branche gauche → Bloc de branche gauche)
   Max       : 0.6032  (ample → Apex)

   🟢 Excellent (sim ≥ 0.85) : 43 termes (22%)
   🟡 Bon      (0.70–0.85)   : 66 termes (34%)
   🟠 Moyen    (0.50–0.70)   : 75 termes (38%)
   🔴 Faible   (sim < 0.50)  : 11 termes (6%)



🎯 Carte des distances tracée.
   Chaque ligne relie un terme étudiant (⭐) à son concept ontologie le + proche (●).
   🟢 Vert = proche (bon match)  →  🔴 Rouge = loin (terme non couvert par l'ontologie)


In [7]:

# ============================================================
# CELLULE 7 — Tableau Détaillé : Meilleurs & Pires Matchs
# ============================================================

df_display = df_nn[['terme_etudiant', 'concept_onto', 'ontology_id', 'categorie', 'cosine_similarity']].copy()
df_display = df_display.rename(columns={
    'terme_etudiant':    '⭐ Terme Étudiant',
    'concept_onto':      '🔵 Concept Ontologie',
    'ontology_id':       'ID OWL',
    'categorie':         'Catégorie',
    'cosine_similarity': 'Sim. Cosinus',
})
df_display['Sim. Cosinus'] = df_display['Sim. Cosinus'].round(4)
df_sorted = df_display.sort_values('Sim. Cosinus', ascending=True)

print("=" * 80)
print("🔴 TOP 20 — Termes étudiants les PLUS ÉLOIGNÉS de l'ontologie")
print("   (= termes mal couverts, potentiels trous dans l'ontologie)")
print("=" * 80)
display(df_sorted.head(40).reset_index(drop=True))

print("\n" + "=" * 80)
print("🟢 TOP 20 — Termes étudiants les PLUS PROCHES de l'ontologie")
print("   (= excellent alignement sémantique)")
print("=" * 80)
display(df_sorted.tail(20).sort_values('Sim. Cosinus', ascending=False).reset_index(drop=True))

# ─── Histogramme de distribution des similarités ──────────────
fig3 = go.Figure()
fig3.add_trace(go.Histogram(
    x=df_nn['cosine_similarity'],
    nbinsx=30,
    marker_color='#2196F3',
    opacity=0.8,
))

# Lignes de seuils
for thresh, label, color in [(0.85, 'Excellent', '#4CAF50'), (0.70, 'Bon', '#FF9800'), (0.50, 'Moyen', '#F44336')]:
    fig3.add_vline(x=thresh, line=dict(color=color, dash='dash', width=2),
                   annotation_text=label, annotation_position="top")

fig3.update_layout(
    title=dict(text="📊 Distribution des Similarités Cosinus (Étudiant → Concept le + proche)", 
               font=dict(size=14, color='#e0e0e0'), x=0.5),
    xaxis=dict(title="Similarité Cosinus", range=[0.2, 1.05]),
    yaxis=dict(title="Nombre de termes"),
    plot_bgcolor='#1e1e1e', paper_bgcolor='#1e1e1e',
    font=dict(color='#e0e0e0'),
    width=900, height=400,
    bargap=0.05,
)
fig3.show()

print(f"\n💡 Les termes 🔴 en bas du tableau sont des candidats pour enrichir l'ontologie.")


🔴 TOP 20 — Termes étudiants les PLUS ÉLOIGNÉS de l'ontologie
   (= termes mal couverts, potentiels trous dans l'ontologie)


,⭐ Terme Étudiant,🔵 Concept Ontologie,ID OWL,Catégorie,Sim. Cosinus
0,ample,Apex,APEX,DESCRIPTEUR_ECG,0.3968
1,62 bpm,Tachycardie,TACHYCARDIE,DESCRIPTEUR_ECG,0.4132
2,assez fines,ST moins,COURANT_DE_LÉSION_SOUS_ENDOCARDIQUE,SIGNE_ECG_PATHOLOGIQUE,0.4135
3,78 bpm,Tachycardie,TACHYCARDIE,DESCRIPTEUR_ECG,0.4156
4,62bpm,Rythme sinusal,RYTHME_SINUSAL,SIGNE_ECG_PATHOLOGIQUE,0.4446
5,HAG limite,HVG,HYPERTROPHIE_VENTRICULAIRE_GAUCHE,DIAGNOSTIC_MAJEUR,0.4631
6,75 bpm,Tachycardie,TACHYCARDIE,DESCRIPTEUR_ECG,0.4719
7,symétrique,TJ orthodromique utilisant une voie accessoire,TJ_ORTHODROMIQUE_UTILISANT_UNE_VOIE_ACCESSOIRE,DIAGNOSTIC_MAJEUR,0.4727
8,conduction 4/1,Faisceau accessoire à conduction antérograde,FAISCEAU_ACCESSOIRE_À_CONDUCTION_ANTÉROGRADE,DIAGNOSTIC_MAJEUR,0.4765
9,70 bpm,Tachycardie,TACHYCARDIE,DESCRIPTEUR_ECG,0.4851



🟢 TOP 20 — Termes étudiants les PLUS PROCHES de l'ontologie
   (= excellent alignement sémantique)


,⭐ Terme Étudiant,🔵 Concept Ontologie,ID OWL,Catégorie,Sim. Cosinus
0,Rythme atrial électroentrainé,Rythme atrial électroentrainé,STIMULATION_ATRIALE,DESCRIPTEUR_ECG,1.0000
1,QRS fins,QRS fins,QRS_FINS,DESCRIPTEUR_ECG,1.0000
2,BAV complet,BAV complet,BAV_COMPLET,DIAGNOSTIC_URGENT,1.0000
3,Rythme sinusal,Rythme sinusal,RYTHME_SINUSAL,SIGNE_ECG_PATHOLOGIQUE,1.0000
4,Bloc de branche gauche,Bloc de branche gauche,BLOC_DE_BRANCHE_GAUCHE,SIGNE_ECG_PATHOLOGIQUE,1.0000
5,Fibrillation atriale,Fibrillation atriale,FIBRILLATION_ATRIALE,DIAGNOSTIC_MAJEUR,1.0000
6,BAV 1,BAV 1,BAV_DE_TYPE_1,DESCRIPTEUR_ECG,1.0000
7,Axe normal,Axe normal,AXE_NORMAL,DESCRIPTEUR_ECG,1.0000
8,BBD complet,BBD complet,BLOC_DE_BRANCHE_DROIT_COMPLET,DIAGNOSTIC_MAJEUR,1.0000
9,BBG,BBG,BLOC_DE_BRANCHE_GAUCHE,SIGNE_ECG_PATHOLOGIQUE,1.0000



💡 Les termes 🔴 en bas du tableau sont des candidats pour enrichir l'ontologie.
